In [ ]:
# 1. Install required libraries
!pip install -q -U bitsandbytes transformers accelerate peft trl datasets

import torch
import gc
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

# --- GPU Memory Cleanup ---
def clear_memory():
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
clear_memory()

# --- 1. Data Preparation ---
print("--- Loading and tokenizing data ---")
model_id = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

df_train = pd.read_csv('training_data.csv')
df_inference = pd.read_csv('dataset_clean.csv')

def tokenize_function(row):
    text = f"<s>[INST] Du bist ein Experte für österreichisches Steuerrecht. Beantworte die folgende Frage präzise: {row['prompt']} [/INST] {row['answer']} </s>"
    return tokenizer(text, truncation=True, max_length=256, padding="max_length") # Increased max_length for L4

train_ds = Dataset.from_list([tokenize_function(row) for _, row in df_train.iterrows()])

# --- 2. Model Loading (4-bit Optimized for L4) ---
print(f"--- Loading {model_id} for L4 GPU ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # L4 supports BF16
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa" # Native Flash Attention support
)

# Prepare for k-bit training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# --- 3. LoRA Configuration ---
peft_config = LoraConfig(
    r=16, # Higher rank possible on L4
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# --- 4. Training Configuration (L4 Native) ---
training_args = TrainingArguments(
    output_dir="./mistral_results",
    per_device_train_batch_size=4, # Higher batch size for L4
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    bf16=True, # L4 advantage: Use BF16 instead of FP16
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    max_grad_norm=1.0
)

# --- 5. Fine-Tuning ---
print("--- Starting Fine-Tuning ---")
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    peft_config=peft_config,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

# --- 6. Inference & CSV Output ---
print("--- Running Inference on L4 --- ")
model.eval()
results = []

for i, row in tqdm(df_inference.iterrows(), total=len(df_inference)):
    prompt = f"<s>[INST] Du bist ein Experte für österreichisches Steuerrecht. Beantworte die folgende Frage präzise: {row['prompt']} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.split("[/INST]")[-1].strip()
    results.append({"id": row['id'], "answer": answer})

    # Erstelle Test-CSV nach 5 Antworten
    if len(results) == 5:
        pd.DataFrame(results).to_csv('test_output_5.csv', index=False)
        print("\n--- Test-Datei 'test_output_5.csv' wurde nach 5 Antworten erstellt. ---")

output_df = pd.DataFrame(results)
output_df.to_csv('fine_tune_output.csv', index=False)
print("\n✅ Process complete on L4. Final result saved to 'fine_tune_output.csv'.")

--- Loading and tokenizing data ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


--- Loading mistralai/Mistral-7B-v0.1 for L4 GPU ---


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Step,Training Loss


--- Running Inference on L4 --- 


  1%|          | 5/643 [01:13<2:35:36, 14.63s/it]


--- Test-Datei 'test_output_5.csv' wurde nach 5 Antworten erstellt. ---


 99%|█████████▉| 638/643 [2:34:51<01:12, 14.54s/it]